# 🎯 [C202_M06] 대역폭 사용률, 큐 대기 현황 및 패킷 유실률 실시간 모니터링 UI 구현
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m06_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M06 |
| **미션 주제** | 대역폭 사용률, 큐 대기 현황 및 패킷 유실률 실시간 모니터링 UI 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 대역폭 사용률, 큐 대기 현황 및 패킷 유실률 실시간 모니터링 UI 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

대역폭 사용률, 큐 대기 전문 현황, 패킷 손실률을 실시간 표출하는 통신 관제 UI를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **대역폭 사용률, 긴급도별 큐 대기 전문, 패킷 손실률, 기한 임박·초과 경보를 1초 주기로 보여 주는 통신 관제 화면을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **재밍과 대역폭 감소가 포함된 20분 운용 기록을 1초 주기로 읽어 관제 지표를 갱신하고 처리시간을 측정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/c202_dashboard.py (또는 웹 UI 컴포넌트)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 작전 상황판으로서의 직관성 및 인지 부하 감소도

#### 미션 PASS 조건

- 정량 기준: 화면 갱신 주기 1초 이내, UI 응답성 100ms 이내


- 진행 기준: 주요 지표 시각화 컴포넌트 4종 이상 렌더링 및 실시간 데이터 연동


- 검증 방법: 사용자 인터랙션 테스트 및 모의 데이터 스트림 렌더링 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""전송 시뮬레이션 기록을 1초 주기로 읽어 대역폭 사용률, 긴급도별 큐 대기 전문 수, 패킷 손실률, 기한 임박·초과 경보를 표출하는 통신 관제 화면(야간 모드)을 구현하고 화면 갱신 처리시간을 측정한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

# ---- 우선순위 큐·재전송 이산 사건 시뮬레이터 공통 함수 ----
HDR = 32   # 전문 조각(패킷) 헤더 bytes (미션 2의 헤더 압축 적용 시 12 bytes)


def policy_key(m, t, policy, aging):
    if policy == "fifo":                                   # 도착 순서
        return (m["t0"],)
    if policy == "strict":                                 # 긴급도 우선, 같으면 도착 순서
        return (RANK[m["prec"]], m["t0"])
    # aging: 긴급도 + (대기시간 / 전달 목표시간) 가중. 오래 기다린 전문과 기한이 임박한 전문이 앞으로 온다.
    return (RANK[m["prec"]] - aging * (t - m["t0"]) / DEADLINE[m["prec"]], m["t0"])


def simulate(msgs, link, policy="strict", mtu=256, preempt=True, rto_mode="fixed", rto_s=4.0, aging=0.0,
             redundancy=False, adapt_mtu=False, hdr=HDR, seed=0, until=None, admit=None):
    """전문 목록을 링크로 보내는 이산 사건 시뮬레이션.
    - 전문은 mtu 크기 조각으로 나눠 보낸다. 조각이 손실되면 RTO 후 재전송한다(자료는 버리지 않음).
    - preempt=False면 한 전문의 조각을 끝까지 연속 전송한다(큰 전문이 긴급 전문을 막는 현상 재현).
    - rto_mode='adaptive'면 왕복시간(RTT) 추정값으로 RTO를 계산한다(Jacobson 방식).
    - redundancy=True면 링크 열화 추정 시 FLASH·IMMEDIATE 조각을 2회 연속 전송한다.
    - adapt_mtu=True면 링크 열화 추정 시 조각 크기를 줄인다.
    - admit(m, t)이 False인 전문은 새 조각을 보내지 않고 저장해 둔다(성능 저하 모드)."""
    rng = np.random.default_rng(seed)
    pend = sorted(({**m, "unsent": m["size"], "ready": [], "frags": [], "done": None, "first": None} for m in msgs), key=lambda m: m["t0"])
    t, i, active, cur = 0.0, 0, [], None
    srtt, rttvar, loss = None, None, 0.0
    st = {"tx": 0, "retx": 0, "spurious": 0, "bytes": 0, "decide_ms": [], "flag": {}, "log": []}
    until = until if until is not None else (pend[-1]["t0"] if pend else 0) + 6000
    while (i < len(pend) or active) and t < until:
        while i < len(pend) and pend[i]["t0"] <= t:
            active.append(pend[i]); i += 1
        td = time.perf_counter()
        for m in active:                                   # ACK가 이미 도착한 재전송 예약은 취소
            m["ready"] = [e for e in m["ready"] if not (e[1]["ack"] is not None and e[1]["ack"] <= t)]
        ready = [m for m in active if any(r <= t for r, _ in m["ready"]) or (m["unsent"] > 0 and (admit is None or admit(m, t)))]
        if not ready:
            nxt = ([pend[i]["t0"]] if i < len(pend) else []) + [r for m in active for r, _ in m["ready"] if r > t]
            t = min(nxt) if nxt else t + 1.0
            continue
        if not preempt and cur in ready and cur["unsent"] > 0:
            m = cur
        else:
            m = min(ready, key=lambda x: policy_key(x, t, policy, aging))
        st["decide_ms"].append((time.perf_counter() - td) * 1000)
        due = [e for e in m["ready"] if e[0] <= t]
        if due:
            e = min(due, key=lambda x: x[0]); m["ready"].remove(e); frag = e[1]
            st["retx"] += 1; st["spurious"] += frag["recv"]
        else:
            size = mtu if not adapt_mtu else (mtu if loss < 0.15 else (mtu // 2 if loss < 0.4 else mtu // 4))
            size = min(m["unsent"], size); m["unsent"] -= size
            frag = {"b": size, "recv": False, "ack": None, "tries": 0}
            m["frags"].append(frag)
        cur = m
        m["first"] = t if m["first"] is None else m["first"]
        nb = frag["b"] + hdr
        copies = 2 if (redundancy and RANK[m["prec"]] <= 1 and loss > 0.15) else 1
        for _ in range(copies):
            t += nb * 8 / link.bw(t)
            st["tx"] += 1; st["bytes"] += nb
            ok = rng.random() >= link.p_loss(nb, t)
            st["log"].append((t, nb, ok, m["prec"], m["id"]))            # 전송 기록(시각, bytes, 성공, 긴급도, 전문)
            loss = 0.85 * loss + 0.15 * (0.0 if ok else 1.0)          # 최근 손실률 추정(EWMA)
            st["flag"][int(t)] = st["flag"].get(int(t), False) or loss > 0.12   # 링크 열화 판정(초 단위)
            if ok and not frag["recv"]:
                frag["recv"], frag["rt"] = True, t
                r = link.rtt(rng); frag["ack"] = t + r
                if srtt is None:
                    srtt, rttvar = r, r / 2
                else:
                    rttvar = 0.75 * rttvar + 0.25 * abs(srtt - r); srtt = 0.875 * srtt + 0.125 * r
        frag["tries"] += 1
        rto = rto_s if (rto_mode == "fixed" or srtt is None) else float(np.clip(srtt + 4 * rttvar, 1.0, 30.0))
        if not frag["recv"] or frag["ack"] > t + rto:                   # 손실 또는 ACK 지연 -> 재전송 예약
            m["ready"].append((t + rto, frag))
        if m["done"] is None and m["unsent"] == 0 and all(f["recv"] for f in m["frags"]):
            m["done"] = max(f["rt"] for f in m["frags"])
        active = [x for x in active if not (x["unsent"] == 0 and not x["ready"] and all(f["recv"] for f in x["frags"]))]
    res = [{"id": m["id"], "type": m["type"], "prec": m["prec"], "size": m["size"], "t0": m["t0"], "done": m["done"], "wait": None if m["first"] is None else m["first"] - m["t0"],
            "retx": sum(f["tries"] for f in m["frags"]) - len(m["frags"]),
            "latency": None if m["done"] is None else m["done"] - m["t0"]} for m in pend]
    for r in res:
        r["on_time"] = r["latency"] is not None and r["latency"] <= DEADLINE[r["prec"]]
    return res, st


def summarize(res):
    """긴급도별 기한 내 전달률·지연. 종합 정확도 = 긴급도별 기한 내 전달률의 평균."""
    df = pd.DataFrame(res)
    rows = []
    for p in PREC:
        d = df[df["prec"] == p]
        lat = d["latency"].dropna()
        rows.append({"긴급도": p, "건수": len(d), "기한내 전달률": round(d["on_time"].mean(), 3) if len(d) else np.nan,
                     "평균 지연(s)": round(lat.mean(), 1) if len(lat) else np.nan,
                     "95% 지연(s)": round(lat.quantile(0.95), 1) if len(lat) else np.nan, "미전달": int(d["done"].isna().sum())})
    tab = pd.DataFrame(rows)
    return tab, round(float(tab["기한내 전달률"].mean()), 3)

COLORS = {"FLASH": "red", "IMMEDIATE": "orange", "PRIORITY": "deepskyblue", "ROUTINE": "gray"}

# [교육용가상자료] 20분 운용: 300~360초 재밍, 600~800초 대역폭 50% 감소
msgs = make_messages(1200, 0.21, seed=7)
link = Link(12000, seed=1, jam=[(300, 360)], bw_drop=[(600, 800, 0.5)])
res, st = simulate(msgs, link, "aging", aging=0.5, rto_mode="adaptive", redundancy=True, adapt_mtu=True)
log = pd.DataFrame(st["log"], columns=["t", "bytes", "ok", "prec", "id"])
done = {r["id"]: r["done"] for r in res}


def snapshot(now, window=10):
    """now 시각의 관제 지표: 최근 window초 사용률·손실률, 긴급도별 대기 전문, 경보 목록."""
    w = log[(log["t"] > now - window) & (log["t"] <= now)]
    util = w["bytes"].sum() * 8 / (link.bw(now) * window)
    loss = 1 - w["ok"].mean() if len(w) else 0.0
    waiting = [m for m in msgs if m["t0"] <= now and (done[m["id"]] is None or done[m["id"]] > now)]
    depth = {p: sum(m["prec"] == p for m in waiting) for p in PREC}
    alerts = []
    for m in waiting:
        age = now - m["t0"]
        if age > DEADLINE[m["prec"]]:
            alerts.append(("기한초과", m["id"], m["prec"], round(age, 1)))
        elif RANK[m["prec"]] <= 1 and age > 0.7 * DEADLINE[m["prec"]]:
            alerts.append(("기한임박", m["id"], m["prec"], round(age, 1)))
    if loss > 0.3:
        alerts.append(("링크열화", "-", f"손실률 {loss:.0%}", round(now, 0)))
    return {"t": now, "util": min(util, 1.0), "loss": loss, **depth, "alerts": alerts}


# 1) 1초 주기 갱신 모의(1,200회) + 처리시간 측정
frames, ms = [], []
for now in range(1, 1201):
    t0 = time.perf_counter()
    frames.append(snapshot(float(now)))
    ms.append((time.perf_counter() - t0) * 1000)
ts = pd.DataFrame([{k: v for k, v in f.items() if k != "alerts"} for f in frames])
print(f"갱신 1,200회: 처리시간 평균 {np.mean(ms):.2f} ms, 최대 {np.max(ms):.2f} ms (목표: 주기 1초 이내, 응답 100 ms 이내 -> {'달성' if np.max(ms) < 100 else '미달'})")

# 2) 구간별 상황 요약
ts["구간"] = pd.cut(ts["t"], [0, 300, 360, 600, 800, 1200], labels=["평시1", "재밍", "평시2", "대역폭50%", "평시3"])
summary = ts.groupby("구간", observed=True).agg(사용률=("util", "mean"), 손실률=("loss", "mean"), FLASH대기=("FLASH", "max"),
                                                IMMEDIATE대기=("IMMEDIATE", "max"), ROUTINE대기=("ROUTINE", "max")).round(2)
print("\n[구간별 관제 지표: 평균 사용률·손실률, 최대 대기 전문 수]")
print(summary.to_string())

# 3) 경보 패널(긴급 경보 우선)
all_alerts = pd.DataFrame([(f["t"], *a) for f in frames for a in f["alerts"]], columns=["시각", "경보", "전문", "긴급도/내용", "경과"])
first = all_alerts.drop_duplicates(["경보", "전문"]).copy()
print(f"\n경보 발생(중복 제외) {len(first)}건:", first["경보"].value_counts().to_dict())
order = {"기한초과": 0, "링크열화": 1, "기한임박": 2}
print(first.sort_values(by=["경보", "시각"], key=lambda c: c.map(order) if c.name == "경보" else c).head(8).to_string(index=False))

# 4) 관제 화면(야간 모드, 구성요소 5종)
plt.style.use("dark_background")
fig = plt.figure(figsize=(13, 8))
a1 = fig.add_subplot(2, 2, 1); a1.plot(ts["t"], ts["util"] * 100, c="lime"); a1.set_title("Bandwidth utilization (%)"); a1.set_ylim(0, 105)
a2 = fig.add_subplot(2, 2, 2)
a2.stackplot(ts["t"], *[ts[p] for p in PREC], colors=[COLORS[p] for p in PREC], labels=PREC); a2.legend(loc="upper left", fontsize=8)
a2.set_title("Queued messages by precedence")
a3 = fig.add_subplot(2, 2, 3); a3.plot(ts["t"], ts["loss"] * 100, c="orange"); a3.axhline(30, c="red", ls="--")
a3.fill_between(ts["t"], 0, 100, where=ts["loss"] > 0.3, color="red", alpha=0.25); a3.set_title("Packet loss rate (%) - red = link degraded")
for a in (a1, a2, a3):
    a.axvspan(300, 360, color="purple", alpha=0.2); a.axvspan(600, 800, color="yellow", alpha=0.08); a.set_xlabel("time (s)")
a4 = fig.add_subplot(2, 2, 4); a4.axis("off")
last = frames[-1]
kpi = summarize(res)[0]
txt = "\n".join(f"{r['긴급도']:<10} on-time {r['기한내 전달률']:.0%}  p95 {r['95% 지연(s)']:>6}s" for _, r in kpi.iterrows())
a4.text(0, 0.95, f"KPI (t={last['t']:.0f}s)\n{txt}\n\nAlerts total: {len(first)}\n  overdue: {(first['경보'] == '기한초과').sum()}\n"
        f"  link degraded: {(first['경보'] == '링크열화').sum()}", family="monospace", fontsize=11, va="top", color="white")
fig.suptitle("Tactical link monitor (night mode) - training data only", color="white")
plt.tight_layout(); plt.show()
print("\n화면 구성: 사용률 추이, 긴급도별 큐 대기, 손실률(열화 구간 강조), KPI·경보 패널, 재밍/대역폭 감소 구간 표시")


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
갱신 1,200회: 처리시간 평균 0.76 ms, 최대 3.48 ms (목표: 주기 1초 이내, 응답 100 ms 이내 -> 달성)

[구간별 관제 지표: 평균 사용률·손실률, 최대 대기 전문 수]
         사용률   손실률  FLASH대기  IMMEDIATE대기  ROUTINE대기
구간                                                 
평시1     0.84  0.03        1            1          9
재밍      0.99  0.58        1            0         13
평시2     0.99  0.06        1            1         17
대역폭50%  0.97  0.06        2            2         21
평시3     0.98  0.05        1            2         27

경보 발생(중복 제외) 4건: {'기한임박': 2, '링크열화': 1, '기한초과': 1}
   시각   경보    전문    긴급도/내용    경과
805.0 기한초과 M0178     FLASH  10.1
303.0 링크열화     -   손실률 36% 303.0
780.0 기한임박 M0165 IMMEDIATE  21.7
802.0 기한임박 M0178     FLASH   7.1

화면 구성: 사용률 추이, 긴급도별 큐 대기, 손실률(열화 구간 강조), KPI·경보 패널, 재밍/대역폭 감소 구간 표시
```
</details>
